In [ ]:
# Imports og stier

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

DATA_PATH = Path("../../../outputs/outputs/all_matches_features_final.csv")

In [ ]:
# 12 features — bedste konfiguration fra ablation studie
# fjernet: goal_diff, pressure_def_count_r1m, time_since_last_event_s

FEATURES = [
    "distance_m",
    "angle_rad",
    "pressure_nd_dist_m",
    "pressure_def_count_r2m",
    "shooter_speed_mps",
    "ball_speed_mps",
    "shot_body_part",
    "play_pattern",
    "gk_ball_distance",
    "obstruction_count",
    "gk_lateral_offset",
    "gk_depth",
]

print(f"Antal features: {len(FEATURES)}")

In [3]:
# Indlæs data — 5056 skud, Superliga 2024/25
# Egne mål ikke inkluderetseparat event-type i

df = pd.read_csv(DATA_PATH)
df["is_goal"] = pd.to_numeric(df["is_goal"], errors="coerce")
df = df.loc[df["is_goal"].isin([0, 1])].reset_index(drop=True)

print(f"Totalt antal skud : {len(df)}")
print(f"Mål               : {int(df['is_goal'].sum())} ({df['is_goal'].mean()*100:.1f}%)")
print(f"Kampe             : {df['game_id'].nunique()}")

Totalt antal skud : 5056
Mål               : 601 (11.9%)
Kampe             : 192


In [4]:
# Preprocessing og 80/20 split
# kategoriske variable encodes med pd.factorize
# manglende værdier erstattes med media

feat_names = [f for f in FEATURES if f in df.columns]

def encode(df_sub):
    X = df_sub.copy()
    for col in X.columns:
        if X[col].dtype == object or str(X[col].dtype) == "category":
            codes, _ = pd.factorize(X[col])
            X[col] = codes.astype(float)
        else:
            X[col] = pd.to_numeric(X[col], errors="coerce")
    return X.to_numpy(dtype=float)

X_raw = encode(df[feat_names])
y     = df["is_goal"].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

medians = np.nanmedian(X_train, axis=0)
for j in range(X_train.shape[1]):
    X_train[np.isnan(X_train[:, j]), j] = medians[j]
    X_test[np.isnan(X_test[:, j]), j]   = medians[j]

print(f"Træning : {X_train.shape[0]} skud")
print(f"Test    : {X_test.shape[0]} skud")

Træning : 4044 skud
Test    : 1012 skud


In [5]:
# TabPFN træning og evaluering på testset
# AUC giver diskrimination, log-loss giver  kalibrering

from tabpfn import TabPFNClassifier

tabpfn = TabPFNClassifier(device="cpu", fit_mode="fit_with_cache", ignore_pretraining_limits=True)
tabpfn.fit(X_train, y_train)

y_prob = tabpfn.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
ll  = log_loss(y_test, y_prob)
print(f"TabPFN — AUC: {auc:.4f} | Log-loss: {ll:.4f}")

TabPFN — AUC: 0.8961 | Log-loss: 0.2060


In [6]:
# Forudsig xG for alle 5056 skud ikke kun testset)
# Simuleringen bruger xG for alle kampe (). træningskampe

X_full = encode(df[feat_names])
for j in range(X_full.shape[1]):
    X_full[np.isnan(X_full[:, j]), j] = medians[j]

df["xg"] = tabpfn.predict_proba(X_full)[:, 1]

print(f"xG beregnet for {len(df)} skud")
print(f"Gennemsnitlig xG: {df['xg'].mean():.4f}")

xG beregnet for 5056 skud
Gennemsnitlig xG: 0.1098


In [ ]:
# Bernoulli Monte Carlo — kampniveau
# Hvert skud simuleres uafhængigt: G_i ~ Bernoulli(p_i)
# Total mål per hold = sum af Bernoulli-draws (Poisson-Binomial)

np.random.seed(42)
results = []

for game_id in df["game_id"].unique():
    kamp_df   = df[df["game_id"] == game_id].reset_index(drop=True)
    teams     = kamp_df["team_id"].unique()
    team_a_df = kamp_df[kamp_df["team_id"] == teams[0]]
    team_b_df = kamp_df[kamp_df["team_id"] == teams[1]]

    sim_goals_a, sim_goals_b = [], []

    for _ in range(1000):
        goals_a = (np.random.rand(len(team_a_df)) < team_a_df["xg"].values).sum()
        goals_b = (np.random.rand(len(team_b_df)) < team_b_df["xg"].values).sum()
        sim_goals_a.append(goals_a)
        sim_goals_b.append(goals_b)

    for team, xg_mean, actual in [
        (teams[0], np.mean(sim_goals_a), team_a_df["is_goal"].sum()),
        (teams[1], np.mean(sim_goals_b), team_b_df["is_goal"].sum()),
    ]:
        results.append({
            "game_id":        game_id,
            "team_id":        team,
            "expected_goals": xg_mean,
            "actual_goals":   actual,
        })

results_df = pd.DataFrame(results)
results_df.head()

In [ ]:
# Konkret eksempel — Silkeborg IF vs AaB (game_id 2442554)
# Bruges som illustration i thesis teksten

SILKEBORG_ID = 418
AAB_ID       = 401
GAME_ID      = 2442554

ex_df        = df[df["game_id"] == GAME_ID]
sil_df       = ex_df[ex_df["team_id"] == SILKEBORG_ID]
aab_df       = ex_df[ex_df["team_id"] == AAB_ID]

sim_sil, sim_aab = [], []
for _ in range(1000):
    sim_sil.append((np.random.rand(len(sil_df)) < sil_df["xg"].values).sum())
    sim_aab.append((np.random.rand(len(aab_df)) < aab_df["xg"].values).sum())

print("=== Silkeborg IF vs AaB — konkret eksempel til thesis ===")
print(f"Silkeborg IF — skud: {len(sil_df)} | Simuleret xG: {np.mean(sim_sil):.2f} | Faktiske mål: {int(sil_df['is_goal'].sum())}")
print(f"AaB          — skud: {len(aab_df)} | Simuleret xG: {np.mean(sim_aab):.2f} | Faktiske mål: {int(aab_df['is_goal'].sum())}")

In [8]:
# Sæson-tabel: simuleret xG vs. faktiske mål
# goal_difference > 0 → overperformance, < 0 → underperformance

team_names = {
    239: "Brøndby IF",    272: "Lyngby BK",
    401: "AaB",           418: "Silkeborg IF",
    420: "AGF Aarhus",    547: "Viborg FF",
    569: "FC København",  1000: "FC Midtjylland",
    1943: "Randers FC",   2450: "Vejle BK",
    2592: "FC Nordsjælland", 2827: "SønderjyskE",
}

official_ranking = {
    "FC København": 1,    "FC Midtjylland": 2,
    "Brøndby IF": 3,      "Randers FC": 4,
    "FC Nordsjælland": 5, "AGF Aarhus": 6,
    "Viborg FF": 7,       "Silkeborg IF": 8,
    "SønderjyskE": 9,     "Vejle BK": 10,
    "Lyngby BK": 11,      "AaB": 12,
}

season_table = results_df.groupby("team_id")[["expected_goals", "actual_goals"]].sum()
season_table["team_name"]       = season_table.index.map(team_names)
season_table["official_rank"]   = season_table["team_name"].map(official_ranking)
season_table["goal_difference"] = season_table["actual_goals"] - season_table["expected_goals"]

season_table = (
    season_table
    .sort_values("official_rank")
    [["official_rank", "team_name", "expected_goals", "actual_goals", "goal_difference"]]
    .reset_index(drop=True)
)

season_table

,official_rank,team_name,expected_goals,actual_goals,goal_difference
0,1,FC København,49.642,59,9.358
1,2,FC Midtjylland,53.562,64,10.438
2,3,Brøndby IF,57.156,59,1.844
3,4,Randers FC,50.883,56,5.117
4,5,FC Nordsjælland,48.753,49,0.247
5,6,AGF Aarhus,53.265,51,-2.265
6,7,Viborg FF,51.185,60,8.815
7,8,Silkeborg IF,49.387,60,10.613
8,9,SønderjyskE,40.928,44,3.072
9,10,Vejle BK,33.325,37,3.675


In [ ]:
## diagram Sorteret efter officiel slutstilling nummer 1 -12

season_table = season_table.sort_values("official_rank")

fig, ax = plt.subplots(figsize=(14, 8))
y = np.arange(len(season_table))

ax.barh(y, season_table["actual_goals"],
        color="lightgray", edgecolor="black", height=0.8, label="Actual Goals")
ax.barh(y, season_table["expected_goals"],
        color="red", height=0.5, label="Simulated xG")

labels = [f"{r}. {t}" for r, t in zip(season_table["official_rank"], season_table["team_name"])]
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.invert_yaxis()

for i, v in enumerate(season_table["expected_goals"]):
    ax.text(v - 2.5, i, f"{v:.1f}", va="center", ha="right",
            color="white", fontsize=10, fontweight="bold")

for i, v in enumerate(season_table["actual_goals"]):
    xg_val = season_table["expected_goals"].iloc[i]
    ax.text(max(v, xg_val) + 1.2, i, str(int(v)), va="center", ha="left",
            color="black", fontsize=10, fontweight="bold")

ax.set_xlabel("Goals / Simulated Expected Goals")
ax.set_title("Season Aggregated Simulated Expected Goals vs Actual Goals",
             fontsize=20, fontweight="bold")
ax.legend()
ax.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Print resultater klar til copy-paste i thesis
print("=== KOPIER DISSE TAL TIL THESIS TEKSTEN ===\n")
for _, row in season_table.iterrows():
    diff = row["actual_goals"] - row["expected_goals"]
    direction = "over" if diff > 0 else "under"
    print(f"{row['team_name']:20s} — xG: {row['expected_goals']:.1f} | Faktiske mål: {int(row['actual_goals'])} | Forskel: {diff:+.1f} ({direction}performed)")

In [ ]:

from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

actual   = season_table["actual_goals"].values
expected = season_table["expected_goals"].values

rmse = np.sqrt(mean_squared_error(actual, expected))
mae  = mean_absolute_error(actual, expected)
r, p = pearsonr(actual, expected)

print(f"RMSE        : {rmse:.2f} goals")
print(f"MAE         : {mae:.2f} goals")
print(f"Correlation : r = {r:.3f}, p = {p:.3f}")


In [ ]:
# Defensiv side — xGA = modstanderens simulerede xG i samme kamp

results_df["expected_goals_against"] = np.nan
results_df["actual_goals_conceded"]  = np.nan

for game_id in results_df["game_id"].unique():
    rows = results_df[results_df["game_id"] == game_id]
    if len(rows) != 2:
        continue
    idx_a, idx_b = rows.index[0], rows.index[1]
    results_df.loc[idx_a, "expected_goals_against"] = results_df.loc[idx_b, "expected_goals"]
    results_df.loc[idx_a, "actual_goals_conceded"]  = results_df.loc[idx_b, "actual_goals"]
    results_df.loc[idx_b, "expected_goals_against"] = results_df.loc[idx_a, "expected_goals"]
    results_df.loc[idx_b, "actual_goals_conceded"]  = results_df.loc[idx_a, "actual_goals"]

# Tilføj til season_table
season_defensive = results_df.groupby("team_id")[["expected_goals_against", "actual_goals_conceded"]].sum()
season_table["expected_goals_against"] = season_table.index.map(season_defensive["expected_goals_against"]) if "team_id" in season_table.columns else season_defensive["expected_goals_against"].values
season_table["actual_goals_conceded"]  = season_table.index.map(season_defensive["actual_goals_conceded"]) if "team_id" in season_table.columns else season_defensive["actual_goals_conceded"].values

print("Defensiv sæson-tabel tilføjet ✓")

In [ ]:
# Plot — xGA vs faktiske mål lukket ind (defensiv validering)

fig, ax = plt.subplots(figsize=(14, 8))
y = np.arange(len(season_table))

ax.barh(y, season_table["actual_goals_conceded"],
        color="lightgray", edgecolor="black", height=0.8, label="Actual Goals Conceded")
ax.barh(y, season_table["expected_goals_against"],
        color="steelblue", height=0.5, label="Simulated xGA")

labels = [f"{r}. {t}" for r, t in zip(season_table["official_rank"], season_table["team_name"])]
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.invert_yaxis()

for i, v in enumerate(season_table["expected_goals_against"]):
    ax.text(v - 2.5, i, f"{v:.1f}", va="center", ha="right",
            color="white", fontsize=10, fontweight="bold")

for i, v in enumerate(season_table["actual_goals_conceded"]):
    xga_val = season_table["expected_goals_against"].iloc[i]
    ax.text(max(v, xga_val) + 1.2, i, str(int(v)), va="center", ha="left",
            color="black", fontsize=10, fontweight="bold")

ax.set_xlabel("Goals Conceded / Simulated xGA")
ax.set_title("Season Aggregated Simulated xGA vs Actual Goals Conceded",
             fontsize=20, fontweight="bold")
ax.legend()
ax.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Print defensiv validering — klar til thesis
print("=== DEFENSIV VALIDERING — xGA vs faktiske mål lukket ind ===\n")
for _, row in season_table.iterrows():
    xga  = row["expected_goals_against"]
    conc = row["actual_goals_conceded"]
    diff = conc - xga
    # Defensivt: færre mål ind end forventet = god performance
    direction = "underperformed defensively" if diff > 0 else "overperformed defensively"
    print(f"{row['team_name']:20s} — xGA: {xga:.1f} | Faktiske mål imod: {int(conc)} | Forskel: {diff:+.1f} ({direction})")

print(f"\nTotal xGA i sæsonen : {season_table['expected_goals_against'].sum():.1f}")
print(f"Total mål lukket ind: {int(season_table['actual_goals_conceded'].sum())}")